In [1]:
#Imports

import numpy as np
import flwr as fl
from data_utils import (
    load_cmapss,
    add_train_rul_and_fault,
    normalize_train,
    split_clients,
    create_dataloader
)
from client import HeterogeneousFLClient
from server import KrumStrategy
from test_utils import run_test

import warnings
import logging
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

print("Import completed...")

Import completed...


In [2]:
#Client Factory

from flwr.app import Context

def client_fn(context: Context):
    cid = int(context.node_config["partition-id"])

    client_data = clients[cid]

    trainloader = create_dataloader(
        client_data,
        batch_size=32,
        seq_len=30
    )

    return HeterogeneousFLClient(
        client_id=cid,
        input_dim=client_data["input_dim"],
        trainloader=trainloader,
        mu=0.01
    ).to_client()

In [3]:
#Load and Prepare Data

print("Loading CMAPSS training data...")
df = load_cmapss("data/train_FD001.txt")
df = add_train_rul_and_fault(df)
df = normalize_train(df)
clients = split_clients(df)

print("\nClient distribution:")
print("-" * 60)
for cid, data in clients.items():
    print(
        f"Client {cid}: "
        f"engines={data['df']['engine_id'].nunique()}, "
        f"sensors={data['input_dim']}"
    )

Loading CMAPSS training data...

Client distribution:
------------------------------------------------------------
Client 0: engines=25, sensors=21
Client 1: engines=25, sensors=10
Client 2: engines=25, sensors=15
Client 3: engines=25, sensors=18


In [4]:
#Configure Strategy

print("Configuring Federated Learning...")
print("Client-side: FedProx + Norm Clipping")
print("Server-side: Krum Aggregation")
print("-" * 60)

strategy = KrumStrategy(
    num_malicious=1,
    fraction_fit=1.0,
    min_fit_clients=4,
    min_available_clients=4
)

Configuring Federated Learning...
Client-side: FedProx + Norm Clipping
Server-side: Krum Aggregation
------------------------------------------------------------


In [5]:
#Run Federated Training

import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("Starting Federated Learning training...")
print("-" * 60)

fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=4,
    config=fl.server.ServerConfig(num_rounds=10),
    strategy=strategy
)

print("\nFederated training completed.")
print("Global shared model saved as: global_shared_model.pth")

Starting Federated Learning training...
------------------------------------------------------------

Federated training completed.
Global shared model saved as: global_shared_model.pth


In [6]:
#Krum Aggregation History

print("\nKrum aggregation history:")
print("-" * 60)
for item in strategy.krum_history:
    print(
        f"Round {item['round']} | "
        f"Krum selected client update index: "
        f"{item['selected_client_update_index']}"
    )


Krum aggregation history:
------------------------------------------------------------
Round 1 | Krum selected client update index: 1
Round 2 | Krum selected client update index: 0
Round 3 | Krum selected client update index: 2
Round 4 | Krum selected client update index: 2
Round 5 | Krum selected client update index: 0
Round 6 | Krum selected client update index: 1
Round 7 | Krum selected client update index: 0
Round 8 | Krum selected client update index: 0
Round 9 | Krum selected client update index: 0
Round 10 | Krum selected client update index: 0


In [7]:
#Testing

print("Starting test using test_FD001.txt and RUL_FD001.txt...")
print("-" * 60)

test_path = "data/test_FD001.txt"
rul_path = "data/RUL_FD001.txt"

test_results = run_test(test_path,rul_path)

print("\n")
print("=" * 100)
print("TEST Results")
print("=" * 100)

for client_id, results in test_results.items():

    print(f"\nClient {client_id}")
    print("-" * 100)

    print(
        f"{'Sample':<10}"
        f"{'True RUL':<15}"
        f"{'Pred RUL':<15}"
        f"{'True Fault':<15}"
        f"{'Pred Fault':<15}"
    )

    print("-" * 100)

    # Print ALL predictions
    for i in range(len(results)):

        print(
            f"{i:<10}"
            f"{results.iloc[i]['True_RUL']:<15.2f}"
            f"{results.iloc[i]['Pred_RUL']:<15.2f}"
            f"{int(results.iloc[i]['True_Fault']):<15}"
            f"{int(results.iloc[i]['Pred_Fault']):<15}"
        )

    print("-" * 100)
    print(f"Total Test Samples: {len(results)}")

Starting test using test_FD001.txt and RUL_FD001.txt...
------------------------------------------------------------


TEST Results

Client 0
----------------------------------------------------------------------------------------------------
Sample    True RUL       Pred RUL       True Fault     Pred Fault     
----------------------------------------------------------------------------------------------------
0         144.00         88.93          0              0              
1         127.00         88.93          0              0              
2         140.00         88.93          0              0              
3         120.00         88.93          0              0              
4         172.00         88.93          0              0              
5         137.00         88.93          0              0              
6         119.00         88.93          0              0              
7         178.00         88.93          0              0              
8         110.00 